# DeepPCB Defect Detection with YOLOv8 and PyTorch

This notebook trains a YOLOv8 object detection model for PCB defect detection using your Roboflow-exported DeepPCB dataset.

The dataset zip already contains YOLO labels, so we do not need to convert the annotations. We will still inspect and validate them before training.

**Class order in this dataset**

| ID | Class |
|---:|---|
| 0 | copper |
| 1 | mousebite |
| 2 | open |
| 3 | pin-hole |
| 4 | short |
| 5 | spur |

**What you will see after each step**

- What files/folders changed.
- What the data looks like after the operation.
- What changed in the model after training or validation.


## 1. Install and import the libraries

YOLOv8 from Ultralytics runs on PyTorch internally. This cell installs the simple set of libraries used in the notebook.

Run this once. If everything is already installed, pip will finish quickly.


In [ ]:
%pip install -q ultralytics matplotlib pandas pyyaml pillow


In [ ]:
from pathlib import Path
import gc
import random
import shutil
import zipfile
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())
print("Ultralytics YOLOv8 is ready.")


**What changed?**

Nothing in the dataset or model changed yet. We only installed/imported the tools and checked which training device is available.


## 2. Set project paths

The zip file is in your Downloads folder. The extracted dataset and training runs will be created inside this Codex project folder.

If your zip is moved later, only change `ZIP_PATH`.


In [ ]:
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "outputs" else CWD

ZIP_PATH = Path("/Users/admin/Downloads/DeepPCB.v5i.yolov8.zip")
DATA_DIR = PROJECT_ROOT / "data" / "DeepPCB.v5i.yolov8"
DATA_YAML = DATA_DIR / "data.yaml"
RUNS_DIR = PROJECT_ROOT / "runs"

print("Project root:", PROJECT_ROOT)
print("Dataset zip:", ZIP_PATH)
print("Dataset extract folder:", DATA_DIR)
print("Training output folder:", RUNS_DIR)

assert ZIP_PATH.exists(), f"Zip file not found: {ZIP_PATH}"


**What changed?**

Nothing was written yet. We only created variables that point to the zip, dataset folder, YAML file, and training output folder.


## 3. Inspect the zip before extracting

Before using a dataset, always confirm the folder structure. YOLO expects:

```text
train/images
train/labels
valid/images
valid/labels
test/images
test/labels
data.yaml
```


In [ ]:
with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()

split_counts = Counter()
for name in names:
    parts = name.split("/")
    if len(parts) >= 3 and parts[0] in {"train", "valid", "test"} and parts[1] in {"images", "labels"}:
        if not name.endswith("/"):
            split_counts[f"{parts[0]}/{parts[1]}"] += 1

print("Files inside the zip:")
for key in sorted(split_counts):
    print(f"{key:13s}: {split_counts[key]}")

print("\nTop-level files:")
for name in ["README.dataset.txt", "README.roboflow.txt", "data.yaml"]:
    print("-", name, "found" if name in names else "missing")


**What changed?**

Still nothing changed on disk. We only confirmed that the zip is already in YOLOv8 format and includes train, validation, and test splits.


## 4. Extract the dataset

This cell unzips the dataset into:

```text
data/DeepPCB.v5i.yolov8/
```

Keep `FORCE_REEXTRACT = False` for normal use. Set it to `True` only if you want to delete and recreate the extracted dataset folder.


In [ ]:
FORCE_REEXTRACT = False

if FORCE_REEXTRACT and DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
    print("Removed old extracted dataset:", DATA_DIR)

if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)
    print("Extracted dataset to:", DATA_DIR)
else:
    print("Dataset already extracted:", DATA_DIR)

for path in [DATA_DIR / "train", DATA_DIR / "valid", DATA_DIR / "test", DATA_YAML]:
    print(path.relative_to(PROJECT_ROOT), "exists:", path.exists())


**What changed?**

The zip content is now available as normal folders. The important new folders are `train`, `valid`, and `test`, each with `images` and `labels`.


## 5. Fix `data.yaml` so paths are stable

Roboflow exports often use paths like `../train/images`. Those paths can break depending on where the notebook is launched.

We will rewrite `data.yaml` with an explicit dataset root and simple relative split paths.


In [ ]:
print("Original data.yaml:")
print(DATA_YAML.read_text())

class_names = ["copper", "mousebite", "open", "pin-hole", "short", "spur"]

clean_yaml = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(class_names),
    "names": class_names,
}

with DATA_YAML.open("w") as f:
    yaml.safe_dump(clean_yaml, f, sort_keys=False)

print("\nCleaned data.yaml:")
print(DATA_YAML.read_text())


**What changed?**

`data.yaml` was updated so YOLOv8 can reliably find the dataset from any working directory.

The class order is preserved from the Roboflow YOLO labels:

`0=copper, 1=mousebite, 2=open, 3=pin-hole, 4=short, 5=spur`


## 6. Validate image and label files

Each YOLO image should have a label file with the same filename stem.

Each label row should look like:

```text
class_id x_center y_center width height
```

The four box values are normalized between `0` and `1`.


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def inspect_split(split):
    image_dir = DATA_DIR / split / "images"
    label_dir = DATA_DIR / split / "labels"

    images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
    labels = sorted(label_dir.glob("*.txt"))
    label_stems = {p.stem for p in labels}

    missing_labels = [p.name for p in images if p.stem not in label_stems]
    bad_rows = []
    class_counts = Counter()
    total_boxes = 0

    for label_path in labels:
        text = label_path.read_text().strip()
        for row_number, line in enumerate(text.splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                bad_rows.append((label_path.name, row_number, "wrong number of values", line))
                continue
            try:
                cls = int(float(parts[0]))
                values = [float(x) for x in parts[1:]]
            except ValueError:
                bad_rows.append((label_path.name, row_number, "could not parse numbers", line))
                continue
            if cls < 0 or cls >= len(class_names) or any(v < 0 or v > 1 for v in values):
                bad_rows.append((label_path.name, row_number, "value out of range", line))
            class_counts[cls] += 1
            total_boxes += 1

    return {
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "boxes": total_boxes,
        "missing_label_files": len(missing_labels),
        "bad_label_rows": len(bad_rows),
        "class_counts": class_counts,
        "first_missing_labels": missing_labels[:5],
        "first_bad_rows": bad_rows[:3],
    }

reports = [inspect_split(split) for split in ["train", "valid", "test"]]

summary = pd.DataFrame([
    {
        "split": r["split"],
        "images": r["images"],
        "labels": r["labels"],
        "boxes": r["boxes"],
        "missing_label_files": r["missing_label_files"],
        "bad_label_rows": r["bad_label_rows"],
    }
    for r in reports
])

display(summary)

for r in reports:
    print("\n", r["split"].upper())
    print("Class counts:")
    for class_id, name in enumerate(class_names):
        print(f"  {class_id}: {name:10s} -> {r['class_counts'][class_id]}")
    if r["first_missing_labels"]:
        print("First missing labels:", r["first_missing_labels"])
    if r["first_bad_rows"]:
        print("First bad rows:", r["first_bad_rows"])


**What changed?**

No data was changed in this step. We checked that the image and label files match and that the YOLO label values are valid.


## 7. Plot class distribution

This helps us see whether one defect type is much more common than another. Class imbalance can affect training.


In [ ]:
total_counts = Counter()
for r in reports:
    total_counts.update(r["class_counts"])

plt.figure(figsize=(8, 4))
plt.bar(class_names, [total_counts[i] for i in range(len(class_names))])
plt.title("DeepPCB defect class distribution")
plt.xlabel("Class")
plt.ylabel("Number of boxes")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

print("Total boxes:", sum(total_counts.values()))


**What changed?**

Nothing was modified. We only visualized the annotation distribution so we understand the training data better.


## 8. Visualize YOLO labels on images

Before training, always draw a few labels on top of the images. This catches many dataset mistakes early.


In [ ]:
colors = {
    0: "red",
    1: "orange",
    2: "lime",
    3: "cyan",
    4: "yellow",
    5: "magenta",
}

def draw_yolo_boxes(image_path):
    label_path = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    w, h = image.size

    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            cls, xc, yc, bw, bh = line.split()
            cls = int(float(cls))
            xc, yc, bw, bh = map(float, [xc, yc, bw, bh])

            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            x2 = (xc + bw / 2) * w
            y2 = (yc + bh / 2) * h

            color = colors.get(cls, "white")
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1, max(0, y1 - 12)), class_names[cls], fill=color)

    return image

random.seed(7)
train_images = sorted((DATA_DIR / "train" / "images").glob("*.jpg"))
sample_images = random.sample(train_images, 4)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, image_path in zip(axes.ravel(), sample_images):
    ax.imshow(draw_yolo_boxes(image_path))
    ax.set_title(image_path.name[:45])
    ax.axis("off")

plt.tight_layout()
plt.show()


**What changed?**

Nothing was changed on disk. We displayed the images with their bounding boxes so you can visually confirm that annotations are placed correctly.


## 9. Choose the training device

YOLOv8 can train on CUDA GPU, Apple Silicon MPS, or CPU.

CPU training works, but it will be slow. If you have a GPU, this cell should detect it automatically.


In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Training device:", DEVICE)

if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("CUDA device:", torch.cuda.get_device_name(0))
    print(f"CUDA free memory: {free_bytes / 1024**3:.2f} GB")
    print(f"CUDA total memory: {total_bytes / 1024**3:.2f} GB")


**What changed?**

No model weights changed. We only selected the device that YOLOv8 will use for training and inference.


## 10. Load a YOLOv8 model

We will use `yolov8n.pt`, the small YOLOv8 model. It is fast and good for learning.

- `yolov8n.pt` starts from pretrained weights.
- `yolov8n.yaml` starts from scratch.

Pretrained weights usually train better and faster.


In [ ]:
USE_PRETRAINED = True
MODEL_SOURCE = "yolov8n.pt" if USE_PRETRAINED else "yolov8n.yaml"

model = YOLO(MODEL_SOURCE)

print("Loaded model from:", MODEL_SOURCE)
print("Number of target classes:", len(class_names))
print("Target class names:", class_names)


**What changed?**

A YOLOv8 model object now exists in memory. The dataset has not trained it yet.

If `USE_PRETRAINED=True`, the model begins with general object detection knowledge from pretrained weights. Training will adapt those weights to PCB defects.


## 11. Train YOLOv8 on DeepPCB

For first learning runs, keep the model small and the code simple.

Suggested starting values:

- `EPOCHS = 30` for a quick first run.
- `EPOCHS = 80` or more for better results.
- `IMG_SIZE = 640` because the images are 640 x 640.
- `BATCH = 8` is safer than `16` on small GPUs.
- If CUDA runs out of memory, this cell automatically retries with smaller batches.
- If it still fails at batch `1`, restart the notebook kernel and run again, or switch `DEVICE = "cpu"`.


In [ ]:
EPOCHS = 30
IMG_SIZE = 640
BATCH = 8
RUN_NAME = "deeppcb_yolov8n"

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def is_memory_error(error):
    message = str(error).lower()
    return "out of memory" in message or "memoryallocation" in message

candidate_batches = [BATCH, 4, 2, 1]
train_results = None
used_batch = None

for candidate_batch in candidate_batches:
    try:
        clear_memory()

        # Reloading the model keeps each retry clean after a failed CUDA attempt.
        model = YOLO(MODEL_SOURCE)

        print(f"Trying training with batch={candidate_batch}, image size={IMG_SIZE}, device={DEVICE}")

        train_results = model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=candidate_batch,
            patience=10,
            project=str(RUNS_DIR / "detect"),
            name=RUN_NAME,
            exist_ok=True,
            device=DEVICE,
            workers=2,
            cache=False,
            amp=True,
        )

        used_batch = candidate_batch
        print(f"Training completed with batch={used_batch}")
        break

    except Exception as error:
        clear_memory()
        if is_memory_error(error) and candidate_batch != candidate_batches[-1]:
            print(f"CUDA memory error with batch={candidate_batch}. Retrying smaller batch...")
            continue
        raise

if train_results is None:
    raise RuntimeError("Training did not finish. Try DEVICE='cpu' or reduce IMG_SIZE to 512.")

RUN_DIR = RUNS_DIR / "detect" / RUN_NAME
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"

print("Batch actually used:", used_batch)
print("Training run folder:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS, "exists:", BEST_WEIGHTS.exists())
print("Last weights:", LAST_WEIGHTS, "exists:", LAST_WEIGHTS.exists())


**What changed?**

The model weights were updated using the training images and labels.

YOLOv8 saved:

- `best.pt`: the checkpoint with the best validation performance.
- `last.pt`: the checkpoint from the final epoch.
- `results.csv`: training and validation metrics for each epoch.
- plots such as confusion matrix and PR curves, depending on the run.


## 12. Read the training results

This cell shows how loss and mAP changed during training.


In [ ]:
results_csv = RUN_DIR / "results.csv"

if results_csv.exists():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    display(history.tail())

    metric_columns = [c for c in history.columns if "metrics/" in c]
    loss_columns = [c for c in history.columns if "loss" in c]

    if metric_columns:
        history.plot(x="epoch", y=metric_columns, figsize=(10, 4), title="Validation metrics")
        plt.grid(True)
        plt.show()

    if loss_columns:
        history.plot(x="epoch", y=loss_columns, figsize=(10, 4), title="Training and validation losses")
        plt.grid(True)
        plt.show()
else:
    print("No results.csv found yet. Run the training cell first.")


**What changed?**

Nothing new was trained here. We only read the saved training log to understand how the model improved over epochs.


## 13. Validate the best model

Validation checks performance on images that were not used for gradient updates during training.

The most common object detection metrics are:

- `mAP50`: mean average precision at IoU 0.50.
- `mAP50-95`: stricter mAP averaged across IoU thresholds.
- `precision`: how many predicted defects are correct.
- `recall`: how many real defects were found.


In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
)

print("Validation precision:", float(val_metrics.box.mp))
print("Validation recall:", float(val_metrics.box.mr))
print("Validation mAP50:", float(val_metrics.box.map50))
print("Validation mAP50-95:", float(val_metrics.box.map))


**What changed?**

The model weights did not change. YOLOv8 evaluated the trained `best.pt` checkpoint on the validation split and produced metrics.


## 14. Evaluate on the test split

Use the test split after training decisions are made. It gives a cleaner estimate of final performance.


In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    device=DEVICE,
)

print("Test precision:", float(test_metrics.box.mp))
print("Test recall:", float(test_metrics.box.mr))
print("Test mAP50:", float(test_metrics.box.map50))
print("Test mAP50-95:", float(test_metrics.box.map))


**What changed?**

The model weights still did not change. We measured final model quality on the held-out test images.


## 15. Run predictions on test images

This creates visual prediction images with boxes and confidence scores.


In [ ]:
PREDICT_NAME = "deeppcb_test_predictions"
test_images = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))
sample_for_prediction = test_images[:12]

predictions = best_model.predict(
    source=[str(p) for p in sample_for_prediction],
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR / "predict"),
    name=PREDICT_NAME,
    exist_ok=True,
    device=DEVICE,
)

PREDICT_DIR = Path(predictions[0].save_dir)
print("Prediction images saved to:", PREDICT_DIR)
print("Number of images predicted:", len(sample_for_prediction))


**What changed?**

YOLOv8 created new output images in the prediction folder. These images show detected PCB defects with class names and confidence scores.


## 16. Display saved predictions

This cell opens the prediction images that YOLOv8 saved.


In [ ]:
predicted_images = sorted(PREDICT_DIR.glob("*.jpg"))[:6]

if predicted_images:
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    for ax, image_path in zip(axes.ravel(), predicted_images):
        ax.imshow(Image.open(image_path))
        ax.set_title(image_path.name[:35])
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No saved prediction images found. Run the prediction cell first.")


**What changed?**

Nothing changed here. We only displayed the saved prediction images so you can inspect the model visually.


## 17. Use the trained model on one new PCB image

Change `IMAGE_TO_CHECK` to any PCB image path.


In [ ]:
IMAGE_TO_CHECK = test_images[0]

single_result = best_model.predict(
    source=str(IMAGE_TO_CHECK),
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,
    project=str(RUNS_DIR / "predict"),
    name="single_image_check",
    exist_ok=True,
    device=DEVICE,
)

print("Checked image:", IMAGE_TO_CHECK)
print("Saved result folder:", single_result[0].save_dir)
print("Detected boxes:", len(single_result[0].boxes))


**What changed?**

A new prediction output was saved for one image. This is the same pattern you can use later in an app or inspection pipeline.


## 18. Optional: export the model

Keep training in `.pt` format while experimenting. Export only when you need deployment.

Uncomment the line below if you want ONNX export.


In [ ]:
# exported_path = best_model.export(format="onnx")
# print("Exported model:", exported_path)


**What changed?**

By default, nothing changes because export is commented out.

If you uncomment and run it, YOLOv8 will create an exported model file such as ONNX beside the trained weights.


## Final notes

For a stronger model after the first successful run:

- Increase `EPOCHS`.
- Try `yolov8s.pt` instead of `yolov8n.pt`.
- Keep `IMG_SIZE = 640` because the dataset images are 640 x 640.
- Watch both mAP and visual predictions, especially for tiny defects.

The main trained model file is:

```text
runs/detect/deeppcb_yolov8n/weights/best.pt
```
